# Offline Preprocessing (Standalone)
This notebook is completely standalone. It contains all the required classes and functions directly in the cells. It is ready to run on Kaggle/Colab.

> **Note for Kaggle/Colab users**: Please update the checkpoint paths and dataset paths in the configuration section to point to your uploaded datasets.


In [ ]:
# Required Pip Installs (Uncomment if running on Colab/Kaggle)
# !pip install -q transformers diffusers accelerate opencv-python segment-anything
# !pip install -q git+https://github.com/IDEA-Research/GroundingDINO.git
# !pip install -q git+https://github.com/facebookresearch/detectron2.git


In [ ]:
import os
import sys
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import cv2

# Configuration - UPDATE THESE PATHS FOR KAGGLE/COLAB
person_img_path = "../inputs/test_person.jpg"
output_dir = Path("../outputs/preprocessed")
output_dir.mkdir(parents=True, exist_ok=True)

sam_checkpoint = "../checkpoints/sam_vit_h_4b8939.pth"
densepose_cfg = "../detectron2_repo/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml"
densepose_weights = "../checkpoints/densepose_model.pkl"
densepose_repo = "../detectron2_repo/projects/DensePose"

# Pipeline Config
preserve_objects = ["bag", "cat"] 
preserve_arms = False
category = "Upper-body" # Upper-body, Lower-body, Dress

print("Setup complete.")


## Utilities


In [ ]:
# ==================================================
# utils/memory.py
# ==================================================

"""
GPU memory management utilities.

Provides stage-based VRAM management to allow sequential loading/unloading
of large models on a single GPU without OOM.

Extracted from the offline_preprocessing notebook's clear_memory() pattern.
"""

import gc
from contextlib import contextmanager

import torch


def clear_memory():
    """Force garbage collection and clear CUDA cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


@contextmanager
def gpu_stage(stage_name: str):
    """
    Context manager that clears GPU memory after each pipeline stage.

    Usage:
        with gpu_stage("Person Parsing"):
            model = load_model()
            result = model(input)
            del model
        # VRAM is freed here even if an exception occurs
    """
    print(f"[{stage_name}] Starting...")
    try:
        yield
    finally:
        clear_memory()
        print(f"[{stage_name}] Done. VRAM freed.")


def get_device() -> str:
    """Return 'cuda' if available, else 'cpu'."""
    return "cuda" if torch.cuda.is_available() else "cpu"


def print_vram_usage():
    """Print current VRAM usage (debug helper)."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  VRAM: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    else:
        print("  VRAM: N/A (no CUDA)")



In [ ]:
# ==================================================
# utils/image_utils.py
# ==================================================

"""
Image I/O and conversion utilities.

Consolidated from scattered helpers across both notebooks:
- save_rgb, save_mask from offline notebook cell 1
- ensure_rgb, ensure_gray from app_online.py
- resize_rgb from offline notebook cell 1
"""

from pathlib import Path
from typing import Union

import numpy as np
from PIL import Image


# ─── Image Loading ──────────────────────────────────────────────────

def ensure_rgb(path_or_img: Union[Path, str, Image.Image]) -> Image.Image:
    """Open an image and convert to RGB."""
    if isinstance(path_or_img, Image.Image):
        return path_or_img.convert("RGB")
    return Image.open(path_or_img).convert("RGB")


def ensure_gray(path_or_img: Union[Path, str, Image.Image]) -> Image.Image:
    """Open an image and convert to grayscale."""
    if isinstance(path_or_img, Image.Image):
        return path_or_img.convert("L")
    return Image.open(path_or_img).convert("L")


# ─── Image Saving ───────────────────────────────────────────────────

def save_rgb(img: Image.Image, path: Union[Path, str]):
    """Save an RGB image, creating parent directories if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)


def save_mask(mask: Union[Image.Image, np.ndarray], path: Union[Path, str]):
    """
    Save a mask image (grayscale), handling various input formats:
    - PIL Image → save directly as "L"
    - bool ndarray → convert to 0/255
    - float ndarray with max <= 1 → scale to 0/255
    - uint8 ndarray → save directly
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(mask, Image.Image):
        mask.convert("L").save(path)
        return

    mask = np.array(mask)
    if mask.dtype == bool:
        mask = mask.astype(np.uint8) * 255
    elif mask.max() <= 1:
        mask = mask.astype(np.uint8) * 255
    else:
        mask = mask.astype(np.uint8)

    Image.fromarray(mask).save(path)


# ─── Image Resizing ─────────────────────────────────────────────────

def resize_rgb(img: Image.Image, size: tuple = (768, 1024)) -> Image.Image:
    """Resize an RGB image to (width, height) using bicubic interpolation."""
    return img.resize(size, Image.BICUBIC)


# ─── Validation ──────────────────────────────────────────────────────

def validate_nonempty_mask(
    mask_path: Union[Path, str], min_positive_pixels: int = 100
) -> tuple:
    """
    Check that a mask file exists and has sufficient positive (white) area.

    Returns:
        (ok: bool, message: str)
    """
    mask_path = Path(mask_path)
    if not mask_path.exists():
        return False, "file_not_found"

    arr = np.array(Image.open(mask_path).convert("L"))
    positive = int((arr > 127).sum())
    if positive < min_positive_pixels:
        return False, f"too_small_positive_area={positive}"
    return True, f"positive_area={positive}"



In [ ]:
# ==================================================
# postprocess/mask_utils.py
# ==================================================

"""
Mask morphology and selection utilities.

Extracted from offline_preprocessing notebook cell 2.
Provides mask cleaning, label selection, connected component filtering,
and agnostic image/mask generation for VITON preprocessing.
"""

import cv2
import numpy as np
from PIL import Image


# ─── Morphological Operations ───────────────────────────────────────

def clean_mask(mask: np.ndarray, open_k: int = 5, close_k: int = 7, blur_k: int = 0) -> np.ndarray:
    """
    Clean a binary mask using morphological open/close and optional blur.

    Args:
        mask: Binary mask (0/255 uint8 or bool).
        open_k: Kernel size for morphological opening (removes small noise).
        close_k: Kernel size for morphological closing (fills small holes).
        blur_k: Kernel size for Gaussian blur (0 = disabled).

    Returns:
        Cleaned binary mask (0/255 uint8).
    """
    mask = mask.astype(np.uint8)

    if open_k > 0:
        kernel_open = np.ones((open_k, open_k), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    if close_k > 0:
        kernel_close = np.ones((close_k, close_k), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    if blur_k and blur_k > 1:
        if blur_k % 2 == 0:
            blur_k += 1
        mask = cv2.GaussianBlur(mask, (blur_k, blur_k), 0)

    mask = (mask > 127).astype(np.uint8) * 255
    return mask


def keep_largest_component(mask: np.ndarray) -> np.ndarray:
    """
    Keep only the largest connected component in a binary mask.

    Args:
        mask: Binary mask (0/255 uint8).

    Returns:
        Mask with only the largest connected component.
    """
    mask_bin = (mask > 127).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)

    if num_labels <= 1:
        return (mask_bin * 255).astype(np.uint8)

    largest_idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    largest = (labels == largest_idx).astype(np.uint8) * 255
    return largest


def refine_top_mask(top_mask: np.ndarray, open_k: int = 5, close_k: int = 9) -> np.ndarray:
    """
    Refine a clothing/top mask: clean (Removed keep_largest_component to avoid dropping disconnected sleeves).
    """
    top_mask = clean_mask(top_mask, open_k=open_k, close_k=close_k)
    return top_mask


# ─── Label Selection ────────────────────────────────────────────────

def get_binary_mask_from_labels(label_map: np.ndarray, target_labels: list) -> np.ndarray:
    """Create a binary mask (0/255) from selected labels in a segmentation map."""
    mask = np.isin(label_map, target_labels).astype(np.uint8) * 255
    return mask


def pick_best_top_labels(
    label_map: np.ndarray,
    candidate_label_sets: list,
    min_area_ratio: float = 0.01,
    max_area_ratio: float = 0.55,
) -> tuple:
    """
    Choose the best set of labels for the clothing region based on mask area.

    Iterates through candidate label sets, computes the mask area for each,
    and picks the one with the largest area within acceptable bounds.

    Args:
        label_map: Segmentation map (H, W) with integer labels.
        candidate_label_sets: List of label sets to try, e.g. [[4], [5], [4,5]].
        min_area_ratio: Minimum acceptable mask area / total area.
        max_area_ratio: Maximum acceptable mask area / total area.

    Returns:
        (best_labels: list, best_mask: np.ndarray)
    """
    H, W = label_map.shape
    total_area = H * W

    best_labels = None
    best_mask = None
    best_score = -1

    for labels in candidate_label_sets:
        mask = get_binary_mask_from_labels(label_map, labels)
        mask = clean_mask(mask, open_k=3, close_k=7)
        area_ratio = (mask > 127).sum() / total_area

        if area_ratio < min_area_ratio or area_ratio > max_area_ratio:
            continue

        score = area_ratio
        if score > best_score:
            best_score = score
            best_labels = labels
            best_mask = mask

    if best_labels is None:
        # Fallback to label [4] if no candidate is suitable
        best_labels = [4]
        best_mask = get_binary_mask_from_labels(label_map, best_labels)

    return best_labels, best_mask


# ─── Agnostic Generation ────────────────────────────────────────────

def create_robust_agnostic(
    person_img_pil: Image.Image,
    parsing_map: np.ndarray,
    top_labels: list = None,
    object_mask: np.ndarray = None,
    category: str = "Upper-body",
) -> tuple:
    """
    Create agnostic image and mask for VITON inference.

    Erases the clothing region (+ optionally occluding objects like bags)
    and exposed skin (arms, neck) from the person image, replacing
    them with a neutral gray (127).

    Args:
        person_img_pil: Original person image (RGB PIL).
        parsing_map: Segmentation map from SegFormer (H, W).
        top_labels: Label IDs for the upper-body clothing region.
        object_mask: Optional binary mask (0/255) for occluding objects.
        category: Garment category ("Upper-body", "Lower-body", "Dress").

    Returns:
        (agnostic_img: PIL.Image, agnostic_mask: PIL.Image)
    """
    if top_labels is None:
        top_labels = [4]

    person_np = np.array(person_img_pil)

    # 1. Get clothing mask
    mask_cloth = np.isin(parsing_map, top_labels).astype(np.uint8) * 255

    # 2. Merge object mask (e.g. bag) if provided
    if object_mask is not None:
        mask_object = (object_mask > 127).astype(np.uint8) * 255
        mask_cloth = np.maximum(mask_cloth, mask_object)

    # 3. Get skin mask
    # SegFormer mattmdjaga/segformer_b2_clothes labels:
    # 12 = left-leg, 13 = right-leg, 14 = right-arm, 15 = left-arm
    skin_labels = [14, 15]
    if category == "Lower-body":
        skin_labels = [12, 13]
    elif category == "Dress":
        skin_labels = [12, 13, 14, 15]
        
    mask_skin = np.isin(parsing_map, skin_labels).astype(np.uint8) * 255

    # 4. Combine clothing + skin → dilate to cover edges
    combined_mask = np.maximum(mask_cloth, mask_skin)
    kernel = np.ones((15, 15), np.uint8)
    agnostic_mask_np = cv2.dilate(combined_mask, kernel, iterations=2)

    # Prevent lower-body mask from bleeding up into the upper clothes
    if category == "Lower-body":
        mask_preserve = np.isin(parsing_map, [4]).astype(np.uint8) * 255
        agnostic_mask_np[mask_preserve > 127] = 0

    # 5. PROTECT REGIONS FROM BEING ERASED/DRAWN OVER
    # Protect head/face/hair (labels: 1=Hat, 2=Hair, 3=Sunglasses, 11=Face)
    mask_head = np.isin(parsing_map, [1, 2, 3, 11]).astype(np.uint8) * 255
    agnostic_mask_np[mask_head > 127] = 0

    # 6. Create agnostic image: fill masked regions with neutral gray
    agnostic_np = person_np.copy()
    agnostic_np[agnostic_mask_np > 127] = 127

    return Image.fromarray(agnostic_np), Image.fromarray(agnostic_mask_np)


# ─── Feather Mask (for blending) ────────────────────────────────────

def feather_mask(mask_pil: Image.Image, ksize: int = 15) -> np.ndarray:
    """
    Create a soft-edged alpha mask for blending.

    Args:
        mask_pil: Binary mask (PIL Image).
        ksize: Gaussian blur kernel size for feathering.

    Returns:
        Float alpha mask (H, W, 1) in range [0, 1].
    """
    mask = np.array(mask_pil.convert("L")).astype(np.uint8)
    mask = (mask > 127).astype(np.uint8) * 255

    if ksize % 2 == 0:
        ksize += 1

    mask = cv2.GaussianBlur(mask, (ksize, ksize), 0)
    return (mask.astype(np.float32) / 255.0)[..., None]



## Stage 1 & 2: Object Detection and Segmentation


In [ ]:
# ==================================================
# stages/object_detection.py
# ==================================================

"""
Stage 1: Object Detection using Grounding DINO.

Detects objects (e.g. bags, accessories) in the person image that may
occlude the clothing region. These detections are used by the SAM
segmentation stage to create precise object masks.

Extracted from offline_preprocessing notebook cells 7-8.
Model: IDEA-Research/grounding-dino-tiny (~1GB VRAM)
"""

from typing import Optional

import numpy as np
import torch
from PIL import Image



class ObjectDetectionStage:
    """Grounding DINO object detector for finding occluding objects."""

    MODEL_ID = "IDEA-Research/grounding-dino-tiny"

    def __init__(self, device: Optional[str] = None):
        self.device = device or get_device()
        self.processor = None
        self.model = None

    def load(self):
        """Load Grounding DINO model into VRAM."""
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

        self.processor = AutoProcessor.from_pretrained(self.MODEL_ID)
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(
            self.MODEL_ID
        ).to(self.device)
        print(f"[ObjectDetection] Grounding DINO loaded on {self.device}")

    def unload(self):
        """Free model from VRAM."""
        del self.model, self.processor
        self.model = self.processor = None
        clear_memory()

    def run(
        self,
        person_img: Image.Image,
        text_prompt: str = "bag.",
        threshold: float = 0.3,
    ) -> tuple:
        """
        Detect objects matching the text prompt in the person image.

        Args:
            person_img: Person image (PIL RGB).
            text_prompt: Object description for zero-shot detection.
            threshold: Detection confidence threshold.

        Returns:
            (boxes: np.ndarray or None, scores: np.ndarray, labels: list)
            boxes is None if no objects detected.
        """
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load() first.")

        inputs = self.processor(
            images=person_img, text=text_prompt, return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        results = self.processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            box_threshold=threshold,
            text_threshold=threshold,
            target_sizes=[person_img.size[::-1]],  # (H, W)
        )

        result = results[0]
        boxes = result["boxes"].cpu().numpy()
        scores = result["scores"].cpu().numpy()
        labels = result["labels"]

        if len(boxes) == 0:
            print("[ObjectDetection] No objects detected.")
            return None, scores, labels

        print(f"[ObjectDetection] Detected {len(boxes)} objects: {labels}")
        return boxes, scores, labels



In [ ]:
# ==================================================
# stages/object_segmentation.py
# ==================================================

"""
Stage 2: Object Segmentation using SAM (Segment Anything Model).

Takes bounding boxes from Grounding DINO and produces precise pixel-level
masks for occluding objects. These masks are used to:
1. Exclude the object from agnostic generation (so the model doesn't try to
   replace it with clothing)
2. Restore the object onto the final result via alpha blending

Extracted from offline_preprocessing notebook cells 9-10.
Model: SAM ViT-H (~7GB VRAM)
"""

from typing import Optional

import numpy as np
import torch
from PIL import Image



class ObjectSegmentationStage:
    """SAM-based object segmentation from bounding box prompts."""

    SAM_CHECKPOINT = "sam_vit_h_4b8939.pth"
    SAM_MODEL_TYPE = "vit_h"

    def __init__(self, device: Optional[str] = None, checkpoint_path: Optional[str] = None):
        self.device = device or get_device()
        self.checkpoint_path = checkpoint_path or self.SAM_CHECKPOINT
        self.sam = None
        self.predictor = None

    def load(self):
        """Load SAM model into VRAM."""
        from segment_anything import sam_model_registry, SamPredictor

        self.sam = sam_model_registry[self.SAM_MODEL_TYPE](
            checkpoint=self.checkpoint_path
        )
        self.sam.to(self.device)
        self.predictor = SamPredictor(self.sam)
        print(f"[ObjectSegmentation] SAM {self.SAM_MODEL_TYPE} loaded on {self.device}")

    def unload(self):
        """Free model from VRAM."""
        del self.sam, self.predictor
        self.sam = self.predictor = None
        clear_memory()

    def run(
        self,
        person_img: Image.Image,
        boxes: np.ndarray,
        scores: np.ndarray,
        labels: list,
        target_labels: Optional[list[str]] = None,
        open_k: int = 3,
        close_k: int = 7,
    ) -> Optional[np.ndarray]:
        """
        Segment the best-matching objects using SAM with box prompt.

        Args:
            person_img: Person image (PIL RGB).
            boxes: Detection bounding boxes from Grounding DINO.
            scores: Detection confidence scores.
            labels: Detection labels.
            target_labels: Which labels to segment (e.g. ["bag", "cat"]).
            open_k: Morphological opening kernel size for mask cleanup.
            close_k: Morphological closing kernel size for mask cleanup.

        Returns:
            Clean binary mask (0/255 uint8 ndarray) or None if no match.
        """
        if self.predictor is None:
            raise RuntimeError("Model not loaded. Call load() first.")

        # Find the matching detections for the target labels
        if not target_labels:
            matched_indices = list(range(len(labels)))
        else:
            matched_indices = []
            for target_label in target_labels:
                matched_indices.extend([
                    i for i, label in enumerate(labels)
                    if target_label.lower() in str(label).lower()
                ])
            matched_indices = list(set(matched_indices))

        if not matched_indices:
            print("[ObjectSegmentation] No detection matches target labels.")
            return None

        print(f"[ObjectSegmentation] Matched indices: {matched_indices}")
        for idx in matched_indices:
            print(f"  - label={labels[idx]}, score={float(scores[idx]):.3f}")

        # Set image and predict mask
        self.predictor.set_image(np.array(person_img))

        combined_mask = np.zeros(person_img.size[::-1], dtype=np.uint8)

        for idx in matched_indices:
            box = boxes[idx]
            masks_pred, scores_pred, _ = self.predictor.predict(
                box=box,
                multimask_output=False,
            )

            mask = masks_pred[0].astype(np.uint8)
            mask = (mask > 0).astype(np.uint8) * 255
            combined_mask = np.bitwise_or(combined_mask, mask)

        # Clean up the combined mask
        object_mask = clean_mask(combined_mask, open_k=open_k, close_k=close_k)

        print(f"[ObjectSegmentation] Mask generated: {object_mask.shape}, "
              f"positive area: {(object_mask > 127).sum()}")
        return object_mask



## Stage 3: Person Parsing


In [ ]:
# ==================================================
# stages/person_parsing.py
# ==================================================

"""
Stage 3: Person Parsing using SegFormer.

Segments the person image into semantic regions (clothing, skin, hair, etc.)
using SegFormer B2 fine-tuned on clothing data. The parsing map is then
used to create the agnostic image and mask required by StableVITON.

Extracted from offline_preprocessing notebook cells 12-17.
Model: mattmdjaga/segformer_b2_clothes (~1GB VRAM)

SegFormer Label Map (mattmdjaga/segformer_b2_clothes):
    0: Background, 1: Hat, 2: Hair, 3: Sunglasses, 4: Upper-clothes,
    5: Skirt, 6: Pants, 7: Dress, 8: Belt, 9: Left-shoe, 10: Right-shoe,
    11: Face, 12: Left-leg, 13: Right-leg, 14: Left-arm, 15: Right-arm,
    16: Bag, 17: Scarf
"""

from typing import Optional

import cv2
import numpy as np
import torch
from PIL import Image



# Candidate label sets based on Garment Category
CATEGORY_CANDIDATES = {
    "Upper-body": [[4], [4, 7], [7]],
    "Lower-body": [[6], [5], [5, 6]],
    "Dress": [[7], [4, 7], [4, 5, 6]],
}

PARSING_MODEL_ID = "mattmdjaga/segformer_b2_clothes"


class PersonParsingStage:
    """SegFormer-based person parsing and agnostic image generation."""

    def __init__(self, device: Optional[str] = None):
        self.device = device or get_device()
        self.processor = None
        self.model = None

    def load(self):
        """Load SegFormer model into VRAM."""
        from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

        self.processor = AutoImageProcessor.from_pretrained(PARSING_MODEL_ID)
        self.model = AutoModelForSemanticSegmentation.from_pretrained(
            PARSING_MODEL_ID
        ).to(self.device)
        self.model.eval()
        print(f"[PersonParsing] SegFormer loaded on {self.device}")

    def unload(self):
        """Free model from VRAM."""
        del self.model, self.processor
        self.model = self.processor = None
        clear_memory()

    @torch.no_grad()
    def _run_segformer(self, image_pil: Image.Image) -> np.ndarray:
        """Run SegFormer inference and return the label map."""
        inputs = self.processor(images=image_pil, return_tensors="pt").to(self.device)
        outputs = self.model(**inputs)
        logits = outputs.logits  # [B, C, h, w]

        upsampled = torch.nn.functional.interpolate(
            logits,
            size=image_pil.size[::-1],  # (H, W)
            mode="bilinear",
            align_corners=False,
        )
        pred = upsampled.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
        return pred

    def run(
        self,
        person_img: Image.Image,
        object_mask: Optional[np.ndarray] = None,
        category: str = "Upper-body",
    ) -> dict:
        """
        Parse person image and generate agnostic image/mask.

        Args:
            person_img: Person image (PIL RGB).
            object_mask: Optional binary mask (0/255) of occluding objects
                        to include in the agnostic mask.
            category: Garment category ("Upper-body", "Lower-body", "Dress").

        Returns:
            dict with keys:
                - "parsing_map": np.ndarray (H, W) segmentation map
                - "top_labels": list of selected label IDs
                - "person_top_mask": np.ndarray (0/255) clothing mask
                - "agnostic_img": PIL.Image agnostic person image
                - "agnostic_mask": PIL.Image agnostic mask
        """
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load() first.")

        # 1. Run segmentation
        parsing_map = self._run_segformer(person_img)
        print(f"[PersonParsing] Labels found: {np.unique(parsing_map)}")

        # 2. Select best clothing labels
        top_labels, person_top_mask = pick_best_top_labels(
            parsing_map, CATEGORY_CANDIDATES[category],
            min_area_ratio=0.05, max_area_ratio=0.55,
        )
        person_top_mask = clean_mask(person_top_mask, open_k=5, close_k=15)
        print(f"[PersonParsing] TOP_LABELS selected: {top_labels}")

        # 3. Merge object mask into person_top_mask (V10 patch from online notebook)
        if object_mask is not None:
            object_mask_resized = cv2.resize(
                object_mask,
                (person_top_mask.shape[1], person_top_mask.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )
            # Dilate object mask to fill gaps at clothing-object boundary
            kernel = np.ones((11, 11), np.uint8)
            object_mask_dilated = cv2.dilate(object_mask_resized, kernel, iterations=1)

            person_top_mask = np.maximum(person_top_mask, object_mask_dilated)
            person_top_mask = clean_mask(person_top_mask, open_k=0, close_k=15)

        # 4. Create agnostic image and mask
        agnostic_img, agnostic_mask = create_robust_agnostic(
            person_img, parsing_map,
            top_labels=top_labels,
            object_mask=object_mask,
            category=category,
        )

        return {
            "parsing_map": parsing_map,
            "top_labels": top_labels,
            "person_top_mask": person_top_mask,
            "agnostic_img": agnostic_img,
            "agnostic_mask": agnostic_mask,
        }



## Stage 4: DensePose Estimation


In [ ]:
# ==================================================
# stages/densepose.py
# ==================================================

"""
Stage 4: DensePose estimation using Detectron2.

Generates a DensePose visualization map that serves as a body pose
condition for StableVITON. The output is a colored image where each
body part is rendered with a distinct color on a black background.

Extracted from offline_preprocessing notebook cells 19-21.
Model: Detectron2 DensePose R50-FPN (~2GB VRAM)

Dependencies:
    - detectron2 (built from source)
    - DensePose project from detectron2 repo
"""

from pathlib import Path
from typing import Optional, Union

import cv2
import numpy as np
import torch
from PIL import Image



class DensePoseStage:
    """DensePose body pose estimation for VITON conditioning."""

    # Default paths — these should be overridden for Modal deployment
    DEFAULT_CFG = "densepose_rcnn_R_50_FPN_s1x.yaml"
    DEFAULT_WEIGHTS = "densepose_model.pkl"

    def __init__(
        self,
        device: Optional[str] = None,
        cfg_path: Optional[str] = None,
        weights_path: Optional[str] = None,
        detectron2_densepose_dir: Optional[str] = None,
    ):
        self.device = device or get_device()
        self.cfg_path = cfg_path or self.DEFAULT_CFG
        self.weights_path = weights_path or self.DEFAULT_WEIGHTS
        self.detectron2_densepose_dir = detectron2_densepose_dir
        self.predictor = None

    def load(self):
        """Load DensePose model into VRAM."""
        import sys

        # Add DensePose project to path if specified
        if self.detectron2_densepose_dir:
            densepose_path = str(self.detectron2_densepose_dir)
            if densepose_path not in sys.path:
                sys.path.insert(0, densepose_path)

        from detectron2.config import get_cfg
        from detectron2.engine import DefaultPredictor
        from densepose import add_densepose_config

        cfg = get_cfg()
        add_densepose_config(cfg)
        cfg.merge_from_file(self.cfg_path)
        cfg.MODEL.WEIGHTS = self.weights_path
        cfg.MODEL.DEVICE = self.device

        self.predictor = DefaultPredictor(cfg)
        print(f"[DensePose] Model loaded on {self.device}")

    def unload(self):
        """Free model from VRAM."""
        del self.predictor
        self.predictor = None
        clear_memory()

    def run(self, person_img: Image.Image) -> Image.Image:
        """
        Generate a DensePose visualization for the person image.

        The output is a colored body-part map on a black background,
        using the PARULA colormap. This serves as the pose condition
        for StableVITON inference.

        Args:
            person_img: Person image (PIL RGB).

        Returns:
            DensePose visualization image (PIL RGB).
        """
        if self.predictor is None:
            raise RuntimeError("Model not loaded. Call load() first.")

        from densepose.vis.extractor import DensePoseResultExtractor

        # Convert PIL to BGR for detectron2
        img_cv2 = cv2.cvtColor(np.array(person_img), cv2.COLOR_RGB2BGR)

        with torch.no_grad():
            outputs = self.predictor(img_cv2)

        if "instances" not in outputs or len(outputs["instances"]) == 0:
            raise RuntimeError("No person detected in the image for DensePose.")

        instances = outputs["instances"].to("cpu")

        if not instances.has("pred_densepose"):
            raise RuntimeError(
                "Person detected but DensePose prediction missing. "
                "Check config/weights compatibility."
            )

        # Extract DensePose results
        extractor = DensePoseResultExtractor()
        densepose_data = extractor(instances)
        results, boxes_xywh = densepose_data

        # Render DensePose on black background
        black_bg = np.zeros_like(img_cv2)
        img_h, img_w = black_bg.shape[:2]

        for i, result in enumerate(results):
            box = boxes_xywh[i]
            x, y, w, h = [int(v) for v in box]

            # Body part labels (0-24) → normalized to colormap range
            labels = result.labels.cpu().numpy()
            labels_norm = (labels * 255.0 / 24).astype(np.uint8)
            color_map = cv2.applyColorMap(labels_norm, cv2.COLORMAP_PARULA)

            # Only keep colored pixels where body parts are detected
            mask = (labels > 0)[:, :, None].astype(np.uint8)

            # Safe coordinate clipping
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(img_w, x + w), min(img_h, y + h)

            slice_h, slice_w = y2 - y1, x2 - x1
            color_map_sliced = color_map[:slice_h, :slice_w]
            mask_sliced = mask[:slice_h, :slice_w]

            roi = black_bg[y1:y2, x1:x2]
            black_bg[y1:y2, x1:x2] = roi * (1 - mask_sliced) + color_map_sliced * mask_sliced

        # Convert BGR back to RGB for PIL
        densepose_rgb = cv2.cvtColor(black_bg, cv2.COLOR_BGR2RGB)
        print(f"[DensePose] Visualization generated: {densepose_rgb.shape}")

        return Image.fromarray(densepose_rgb)



## Execution
Now we run the pipeline using the classes defined above.


In [ ]:

person_img = Image.open(person_img_path).convert("RGB")
IMG_SIZE = (768, 1024)
person_img = resize_rgb(person_img, IMG_SIZE)
display(person_img.resize((384, 512)))

object_mask = None
if preserve_objects:
    text_prompt = " . ".join(preserve_objects) + " ."
    print(f"Detecting: {text_prompt}")
    
    with gpu_stage("Stage 1: Object Detection"):
        detector = ObjectDetectionStage()
        detector.load()
        boxes, scores, labels = detector.run(person_img, text_prompt)
        detector.unload()
    
    if boxes is not None and len(boxes) > 0:
        with gpu_stage("Stage 2: Object Segmentation"):
            segmenter = ObjectSegmentationStage(checkpoint_path=sam_checkpoint)
            segmenter.load()
            object_mask = segmenter.run(person_img, boxes, scores, labels, target_labels=preserve_objects)
            segmenter.unload()
            if object_mask is not None:
                display(Image.fromarray(object_mask).resize((384, 512)))

with gpu_stage("Stage 3: Person Parsing"):
    parser = PersonParsingStage()
    parser.load()
    parsing_results = parser.run(person_img, object_mask=object_mask, category=category)
    parser.unload()

parsing_map = parsing_results["parsing_map"]
agnostic_img = parsing_results["agnostic_img"]
agnostic_mask = parsing_results["agnostic_mask"]
top_labels = parsing_results["top_labels"]

display(agnostic_img.resize((384, 512)))
display(agnostic_mask.resize((384, 512)))

with gpu_stage("Stage 4: DensePose"):
    densepose = DensePoseStage(cfg_path=densepose_cfg, weights_path=densepose_weights, detectron2_densepose_dir=densepose_repo)
    densepose.load()
    densepose_img = densepose.run(person_img)
    densepose.unload()

display(densepose_img.resize((384, 512)))

# Save
final_restore_mask = None
if preserve_objects and object_mask is not None:
    final_restore_mask = object_mask.copy()

if category == "Lower-body":
    preserve_arms = True

if preserve_arms:
    arm_mask = np.isin(parsing_map, [14, 15]).astype(np.uint8) * 255
    arm_mask = clean_mask(arm_mask, open_k=3, close_k=5)
    if final_restore_mask is None:
        final_restore_mask = arm_mask
    else:
        final_restore_mask = np.maximum(final_restore_mask, arm_mask)

person_img.save(output_dir / "person_img.png")
agnostic_img.save(output_dir / "agnostic_img.png")
agnostic_mask.save(output_dir / "agnostic_mask.png")
densepose_img.save(output_dir / "densepose_img.png")
np.save(output_dir / "top_labels.npy", top_labels)

if final_restore_mask is not None and (final_restore_mask > 0).any():
    Image.fromarray(final_restore_mask).save(output_dir / "final_restore_mask.png")
    
print("Offline preprocessing completed and saved to disk.")

